# Planning with the final collected ranking adapter

Select a GPU runtime and run the cells in order. Run the frozen **band 4** adapter on the **48 exact cases** from the original planning benchmark. Reuse the saved handcrafted and raw episodes, and use the same goal tokens, world model, CEM configuration, starts and seeds. If the GPU, Torch version or matmul setting differs from the historical run, the report flags it.

The primary comparison is handcrafted versus collected. It reports success and failures on all cases, with an efficiency comparison restricted to the cases both adapters solve. The raw baseline is also displayed. Both benchmark inputs are read from their completed Drive runs.

Each finished episode is saved in a new Drive folder. To resume, set `RESUME_RUN_ID` to the printed value and use the same GPU type and runtime configuration.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RUNS=DRIVE/'datasets/curriculum_cost_adapter'
HISTORICAL_RUN=RUNS/'ranking_planning_v1/20261106T204214364650Z'
COLLECTED_RUN=RUNS/'experience_curriculum_v1/20261107T092616170347Z'
assert (HISTORICAL_RUN/'conditions/adapted/stage.json').is_file(), 'Set HISTORICAL_RUN to the completed planning benchmark.'
assert (COLLECTED_RUN/'band_4/complete.json').is_file(), 'Set COLLECTED_RUN to the completed experience curriculum.'
RESUME_RUN_ID=None
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=RUNS/'experience_planning_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-planning')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='b395e3585a7353d23ab7226bd9569d5776f52d52'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-planning-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Historical planning run:',HISTORICAL_RUN)
print('Collected adapter run:',COLLECTED_RUN)
print('Run ID:',RUN_ID)
print('New planning results:',OUTPUT)

## Load pinned code and frozen models

The notebook verifies the world-model and encoder checkpoint hashes against the saved benchmark.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Run the final collected adapter

The original benchmark used six goals and eight starts per goal. All starts and goals are on the trained side of the wall. Goal 0 was used during fitting, and the other five goals were held out as goal references.

In [ ]:
started=time.monotonic()
shell([sys.executable,'-m','src.mylonas_ijepa_wm.experience_planning',
       '--historical-run',HISTORICAL_RUN,'--collected-run',COLLECTED_RUN,'--output-dir',OUTPUT,
       '--world-model-checkpoint',CHECKPOINT,'--encoder-checkpoint',ENCODER,
       '--dino-wm-root',DINO_ROOT,'--device','cuda'],cwd=REPO)
print('Benchmark minutes:',round((time.monotonic()-started)/60,2))
report=json.loads((OUTPUT/'report.json').read_text())

## Compare the planners

An action is one executed simulator step. Path length is the sum of the distances traveled by those steps. A negative paired delta means the collected adapter used fewer actions or traveled a shorter path. A success can still include an excursion outside the original fitting region.

In [ ]:
import pandas as pd

comparison=report['handcrafted_vs_collected']
against_raw=report['raw_vs_collected']
conditions={
    'raw':against_raw['overall']['conditions']['raw'],
    'handcrafted':comparison['overall']['conditions']['raw'],
    'collected':comparison['overall']['conditions']['adapted'],
}
overview=pd.DataFrame.from_dict(conditions,orient='index').drop(columns='failed_case_ids')
print('All cases')
display(overview.round(3))
print('Saved vs current numerical runtime')
display(pd.DataFrame.from_dict(report['numeric_runtime'],orient='index',columns=['historical','collected']))
print('Both-success efficiency')
display(pd.DataFrame({
    label:summary['overall']['both_success_efficiency']
    for label,summary in [('handcrafted → collected',comparison),('raw → collected',against_raw)]
}).rename(index=lambda key:key.replace('raw_','baseline_').replace('adapted_','collected_')).round(3))
print('Collected-only successes against handcrafted:',comparison['overall']['won_case_ids'])
print('Handcrafted-only successes against collected:',comparison['overall']['lost_case_ids'])

def group_rows(groups):
    return pd.DataFrame.from_dict({
        label:dict(cases=row['episodes'],handcrafted_successes=row['conditions']['raw']['successes'],
                   collected_successes=row['conditions']['adapted']['successes'],
                   both_success=row['both_success'],
                   action_delta=row['both_success_efficiency']['mean_steps_delta'],
                   path_delta=row['both_success_efficiency']['mean_path_length_delta'])
        for label,row in groups.items()
    },orient='index').round(3)

print('By goal')
display(group_rows({str(tuple(row['goal'])):row for row in comparison['by_goal'].values()}))
print('By start-distance band')
display(group_rows(comparison['by_distance_band']))

table=pd.DataFrame([
    dict(cost=method,case=r['case_id'],goal=tuple(r['case']['goal']),start=tuple(r['case']['start']),
         seed=r['case']['seed'],band=r['case']['band'],success=r['success'],actions=r['micro_steps'],
         path_length=r['path_length'],final_distance=r['final_goal_distance'],
         outside_fitting_region=r['ever_outside_fitting'])
    for method,rows in report['records'].items() for r in rows
])
table.to_csv(OUTPUT/'per_case.csv',index=False)
print('Failed cases across all costs')
display(table[~table['success']])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

runs={method:{r['case_id']:r for r in rows} for method,rows in report['records'].items()}
manual,learned=runs['handcrafted'],runs['collected']
disagreements=[k for k in manual if manual[k]['success'] != learned[k]['success']]
shared=[k for k in manual if manual[k]['success'] and learned[k]['success']]
shared.sort(key=lambda k:abs(learned[k]['micro_steps']-manual[k]['micro_steps']),reverse=True)
selected=(disagreements+shared)[:4]
if selected:
    fig,axes=plt.subplots(1,len(selected),figsize=(4*len(selected),6),squeeze=False,constrained_layout=True)
    for ax,key in zip(axes[0],selected):
        case=manual[key]['case']
        for method,color in [('handcrafted','tab:blue'),('collected','tab:orange')]:
            row=runs[method][key]
            xy=np.asarray(row['primitive_positions'])
            status='solved' if row['success'] else 'failed'
            ax.plot(*xy.T,color=color,lw=1.8,label=f"{method} · {row['micro_steps']} actions · {status}")
            ax.scatter(*xy[-1],c=color,s=24)
        ax.add_patch(Circle((50,42),34,fill=False,ls=':',color='gray'))
        ax.scatter(*case['start'],marker='x',c='black',s=55,label='Start')
        ax.scatter(*case['goal'],marker='*',c='black',s=80,label='Goal')
        ax.set(title=key,xlabel='Ball x',ylabel='Ball y',xlim=(4,60),ylim=(4,60),aspect='equal')
        ax.legend(fontsize=7,loc='upper center')
    fig.suptitle('Cases with different outcomes first, then the largest action differences')
    fig.savefig(OUTPUT/'paired_paths.png',dpi=140,bbox_inches='tight')
    plt.show()
    plt.close(fig)